# TrialMatch: Colab training and evaluation

Use this notebook to run the QLoRA training and GPU experiments. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available CUDA GPU) before running the cells. Mount Google Drive so model adapters and outputs survive a runtime disconnect. Training time and availability depend on the assigned GPU; do not treat an interrupted run as a completed experiment.

The held-out test cell is disabled by default. Run it only once all model and prompt decisions are final. Never upload patient data, model weights, or credentials to a public repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO = Path('/content/trialmatch-genai')
DRIVE_RUN = Path('/content/drive/MyDrive/trialmatch-genai')
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Persistent experiment directory:', DRIVE_RUN)

In [ ]:
import subprocess

if not REPO.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/Quttoshi/trialmatch-genai.git', str(REPO)],
        check=True,
    )
else:
    print('Using existing checkout:', REPO)
%cd /content/trialmatch-genai

## Install and verify the GPU

This installs the repository's optional training dependencies into the Colab runtime. If installation fails, record the error and runtime package versions rather than silently changing dependency versions.

In [ ]:
%pip install -e '.[train,test]'

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select a GPU runtime in Colab, then rerun.')
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('BF16 supported:', torch.cuda.is_bf16_supported())
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## Prepare the prescribed split and train

The prepared JSONL files are local to the Colab runtime and are ignored by Git. The model adapter, checkpoints, and logs are written to Drive. If Colab disconnects, remount the same Drive, rerun the setup and data-preparation cells, then rerun the training cell; it resumes from the latest checkpoint if one exists, otherwise it starts a new run. Checkpoints are saved at epoch boundaries, so progress since the latest checkpoint (at most one epoch) must run again. The training command uses the train and validation files only; it does not read the test split.

In [ ]:
import subprocess

subprocess.run(['python', '-m', 'trialmatch.prepare_data', '--output-dir', 'data/processed'], check=True)
print(Path('data/processed/split_manifest.json').read_text())

In [ ]:
ADAPTER_DIR = DRIVE_RUN / 'qwen25-1.5b-qlora'
subprocess.run([
    'python', '-m', 'trialmatch.train',
    '--train-file', 'data/processed/train.jsonl',
    '--validation-file', 'data/processed/validation.jsonl',
    '--output-dir', str(ADAPTER_DIR),
    '--max-seq-len', '1024',
    '--epochs', '3',
    '--resume-from-checkpoint',
], check=True)

In [ ]:
import json

metadata_path = ADAPTER_DIR / 'training_metadata.json'
adapter_path = ADAPTER_DIR / 'adapter'
if not metadata_path.is_file() or not adapter_path.is_dir():
    raise FileNotFoundError('Training metadata or saved adapter is missing; check the training output.')
print(json.dumps(json.loads(metadata_path.read_text()), indent=2))
print('Adapter files:', sorted(path.name for path in adapter_path.iterdir()))

## Compare baseline and fine-tuned validation performance

Both runs use the same validation split and deterministic greedy decoding. Review classification and evidence metrics, invalid outputs, truncation, and example-level predictions before freezing the approach.

In [ ]:
VALIDATION_DIR = DRIVE_RUN / 'validation'
subprocess.run([
    'python', '-m', 'trialmatch.evaluate',
    '--split', 'validation',
    '--adapter', str(ADAPTER_DIR / 'adapter'),
    '--output-dir', str(VALIDATION_DIR),
], check=True)

In [ ]:
for run_name in ('baseline', 'finetuned'):
    metrics_file = VALIDATION_DIR / run_name / 'metrics.json'
    print(f'\n===== {run_name} =====')
    print(metrics_file.read_text())

## Final held-out test (manual opt-in only)

Do not run this cell while selecting prompts, hyperparameters, or other model decisions. First review validation results and freeze all choices. Then set `RUN_FINAL_TEST = True` yourself and run once. This is intentionally not enabled automatically.

In [ ]:
RUN_FINAL_TEST = False  # Change to True only after all decisions are frozen.
if RUN_FINAL_TEST:
    TEST_DIR = DRIVE_RUN / 'final_test'
    subprocess.run([
        'python', '-m', 'trialmatch.evaluate',
        '--split', 'test', '--allow-test',
        '--adapter', str(ADAPTER_DIR / 'adapter'),
        '--output-dir', str(TEST_DIR),
    ], check=True)
else:
    print('Skipped. Freeze all decisions, then explicitly set RUN_FINAL_TEST = True to run once.')

## Optional attention benchmark

This is the project's NumPy CPU reference-vs-tiled benchmark, not a GPU kernel benchmark. It checks numerical agreement and reports theoretical score-tensor sizes; these are not measured process peak-memory values.

In [ ]:
ATTENTION_LOG = DRIVE_RUN / 'attention_benchmark.json'
subprocess.run([
    'python', '-m', 'trialmatch.benchmark_attention',
    '--query-length', '512', '--key-length', '512',
    '--heads', '4', '--head-dim', '64',
    '--query-block-size', '32', '--key-block-size', '32',
    '--repeats', '5', '--output', str(ATTENTION_LOG),
], check=True)
print('Saved:', ATTENTION_LOG)

## Optional KV-cache benchmark

This compares cached decoding with the uncached greedy reference. The longest cases may take substantial time. The JSON includes exact token agreement, latency, throughput, and incremental CUDA allocation.

In [ ]:
KV_LOG = DRIVE_RUN / 'decoding_benchmark.json'
subprocess.run([
    'python', '-m', 'trialmatch.benchmark_decoding',
    '--input-lengths', '128,512,1024',
    '--output-lengths', '16,32',
    '--repeats', '3', '--output', str(KV_LOG),
], check=True)
print('Saved:', KV_LOG)

## Optional speculative-decoding benchmark

Runs a small validation sample and compares speculative outputs to target greedy decoding. Keep the saved JSON even if speculative decoding is slower; a slowdown is a valid result.

In [ ]:
SPECULATIVE_LOG = DRIVE_RUN / 'speculative_benchmark.json'
subprocess.run([
    'python', '-m', 'trialmatch.benchmark_speculative',
    '--validation-file', 'data/processed/validation.jsonl',
    '--target-adapter', str(ADAPTER_DIR / 'adapter'),
    '--examples', '5', '--max-seq-len', '1024',
    '--max-new-tokens', '32', '--draft-token-counts', '2,4,8',
    '--repeats', '3', '--output', str(SPECULATIVE_LOG),
], check=True)
print('Saved:', SPECULATIVE_LOG)

In [ ]:
print('Persistent experiment directory:', DRIVE_RUN)
for path in sorted(DRIVE_RUN.rglob('*')):
    if path.is_file():
        print(path.relative_to(DRIVE_RUN), f'({path.stat().st_size} bytes)')
print('\nCopy measured metrics and benchmark logs into the IEEE report. Include hardware/runtime details, analyze errors, and never invent missing results.')